# CMGAN-George Training Notebook

Dieses Notebook startet das Training von `cmgan-george` direkt aus Jupyter.

Vor dem Start bitte sicherstellen:
- `__config__.py` zeigt auf deine lokalen Datenpfade.
- `models/huBERT_wrapper.py` verweist auf das richtige HuBERT-Modell.
- Die gewünschte Hyperparameter-Datei liegt unter `hparams/`.

## Standard-Setup

Die Zellen unten prüfen zuerst den Projektpfad und starten danach `train.py` mit einer Standard-Konfiguration.

In [ ]:
!pip install speechbrain==1.0.2
!pip install hyperpyyaml
!pip install pesq
!pip install pysepm
!pip install git+https://github.com/schmiph2/pysepm.git
!pip install tensorboard
!pip install onnxruntime
!pip install torchinfo
!pip install pyloudnorm
!pip install glob2
!pip install pyroomacoustics
!pip install einops
!pip install fairseq
!pip install transformers torchaudio
!pip install fairseq

In [ ]:
from pathlib import Path
import os
import sys

candidate_roots = [
    Path.cwd(),
    Path.cwd() / 'cmgan-george',
    Path('/workspaces/MetricGAN-KAN-LwF/cmgan-george'),
]
repo_root = next((path for path in candidate_roots if (path / 'train.py').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Konnte cmgan-george/train.py nicht finden. Bitte das Notebook im Workspace öffnen.')

os.chdir(repo_root)
print(f'Repo root: {repo_root}')
print(f'Python: {sys.executable}')

import sys
hparams_file = Path('hparams/hyperparams_chime_bak_ovr_pesq_1.0.yaml')
if not hparams_file.exists():
    raise FileNotFoundError(f'Hyperparameter-Datei fehlt: {hparams_file}')

print(f'Using hparams: {hparams_file}')

In [ ]:
import shutil
import subprocess
import sys


def parse_pyver(ver_str):
    parts = ver_str.strip().split('.')
    return tuple(int(p) for p in parts[:2])


runner = [sys.executable]
pyver = sys.version.split()[0]
conda = shutil.which('conda')

if conda is not None:
    probe = subprocess.run(
        [conda, 'run', '-n', 'CHIME2023', 'python', '-c', 'import sys; print(sys.version.split()[0])'],
        capture_output=True,
        text=True,
    )
    if probe.returncode == 0:
        runner = [conda, 'run', '-n', 'CHIME2023', 'python']
        pyver = probe.stdout.strip()

if parse_pyver(pyver) >= (3, 10):
    raise RuntimeError(
        f'Aktiver Python ist {pyver}. Diese George-Variante braucht fuer fairseq eine 3.8/3.9-Umgebung. '
        'Bitte CHIME2023-Env aktivieren bzw. Kernel wechseln.'
    )

command = runner + ['train.py', str(hparams_file)]
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys


def parse_pyver(ver_str):
    parts = ver_str.strip().split('.')
    return tuple(int(p) for p in parts[:2])


runner = [sys.executable]
pyver = sys.version.split()[0]
conda = shutil.which('conda')

if conda is not None:
    probe = subprocess.run(
        [conda, 'run', '-n', 'CHIME2023', 'python', '-c', 'import sys; print(sys.version.split()[0])'],
        capture_output=True,
        text=True,
    )
    if probe.returncode == 0:
        runner = [conda, 'run', '-n', 'CHIME2023', 'python']
        pyver = probe.stdout.strip()

if parse_pyver(pyver) >= (3, 10):
    raise RuntimeError(
        f'Aktiver Python ist {pyver}. Diese George-Variante braucht fuer fairseq eine 3.8/3.9-Umgebung. '
        'Bitte CHIME2023-Env aktivieren bzw. Kernel wechseln.'
    )

hparams_dir = Path('hparams')
if not hparams_dir.exists():
    raise FileNotFoundError('Ordner hparams nicht gefunden.')

# Primär: train*.yaml; Fallback: hyperparams*.yaml
hparam_files = sorted(hparams_dir.glob('train*.yaml'))
if not hparam_files:
    hparam_files = sorted(hparams_dir.glob('hyperparams*.yaml'))

if not hparam_files:
    raise RuntimeError('Keine passenden Trainings-YAMLs gefunden (train*.yaml oder hyperparams*.yaml).')

print(f'Gefundene YAMLs: {len(hparam_files)}')
for idx, cfg in enumerate(hparam_files, start=1):
    cmd = runner + ['train.py', str(cfg)]
    print(f'[{idx}/{len(hparam_files)}] ' + ' '.join(cmd))
    subprocess.run(cmd, check=True)

print('Alle Trainingsläufe abgeschlossen.')

## Batch-Run über alle Trainings-YAMLs

Diese Zelle läuft automatisch über alle Konfigurationsdateien in `hparams/`.
Priorität: zuerst `train*.yaml`, falls nichts gefunden wird `hyperparams*.yaml`.